# F-OAL — Table 4 Controlled Architectural Ablation Study (CIFAR-100)

- **Paper:** *F-OAL: Forward-only Online Analytic Learning with Fast Training and Low Memory Footprint in Class Incremental Learning* (NeurIPS 2024)
- **Author Official Repository:** [`https://github.com/liuyuchen-cz/F-OAL`](https://github.com/liuyuchen-cz/F-OAL)
- **Group Repository:** [`https://github.com/caoban123/F-OAL`](https://github.com/caoban123/F-OAL)
- **Target Branch:** `experiment/table4-ablation`
- **Research Question:** Dissect the individual and joint contributions of Feature Fusion (FF) and Smooth Projection (SP) as reported in NeurIPS 2024 Table 4:
  - **A0 (Full F-OAL):** Feature Fusion ON, Smooth Projection ON (Paper: 91.1%, C3-D Reused: 90.40%)
  - **A1 (w/o Feature Fusion):** Feature Fusion OFF, Smooth Projection ON (Paper: 90.6%)
  - **A2 (w/o Smooth Projection):** Feature Fusion ON, Smooth Projection OFF (Paper: 90.7%)
  - **A3 (w/o Both):** Feature Fusion OFF, Smooth Projection OFF (Paper: 90.6%)
- **Strict Controls:**
  - Backbone: `vit_base_patch16_224.augreg2_in21k_ft_in1k` (ImageNet-21K + 1K FT)
  - Preprocessing: Bilinear interpolation, Raw [0, 1] tensor (Official parity from C3-D)
  - Dataset: CIFAR-100, 10 tasks x 10 classes, seed 42
  - Classifier: Online Ridge RLS, batch size 10, gamma 1.0, float64
- **Compute Guard:** A0 is bitwise equivalent to C3-D (`A0_REUSE_APPROVED`), saving 25% GPU compute.

In [ ]:
# === CELL 1: Environment Preflight ===
import os
import sys
import shutil
import platform
import subprocess
import torch

print(f"Python version   : {sys.version}")
print(f"Platform         : {platform.platform()}")
print(f"PyTorch version  : {torch.__version__}")
print(f"CUDA available   : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "FATAL: CUDA GPU is not available! Please switch Colab runtime to GPU before running."
    )

device_name = torch.cuda.get_device_name(0)
device_props = torch.cuda.get_device_properties(0)
vram_gb = device_props.total_memory / (1024 ** 3)
print(f"GPU Device       : {device_name}")
print(f"CUDA version     : {torch.version.cuda}")
print(f"Total VRAM       : {vram_gb:.2f} GB")

free_bytes = shutil.disk_usage(".").free
print(f"Free Disk Space  : {free_bytes / (1024**3):.2f} GB")


In [ ]:
# === CELL 2: Clone Group Repository & Checkout Table 4 Branch ===
from pathlib import Path

REPO_DIR = Path("/content/F-OAL")
BRANCH_NAME = "experiment/table4-ablation"

if REPO_DIR.exists():
    print(f"Removing existing directory {REPO_DIR} for clean clone...")
    shutil.rmtree(REPO_DIR)

subprocess.run([
    "git", "clone", "-b", BRANCH_NAME,
    "https://github.com/caoban123/F-OAL.git", str(REPO_DIR)
], check=True)

commit_hash = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=str(REPO_DIR), text=True
).strip()
print(f"SUCCESS: Repository ready at commit {commit_hash} (branch: {BRANCH_NAME})")

os.chdir(str(REPO_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


In [ ]:
# === CELL 3: Install Dependencies & Setup W&B ===
!pip install -q timm pyyaml pytest
!pip install -q --upgrade wandb

import wandb
WANDB_BASE_URL = "https://api.forge.coreweave.com"
os.environ["WANDB_BASE_URL"] = WANDB_BASE_URL

print(f"Configured WANDB_BASE_URL: {os.environ['WANDB_BASE_URL']}")
print("Please log in to W&B if prompted:")
wandb.login(host=WANDB_BASE_URL)


In [ ]:
# === CELL 4: Prepare CIFAR-100 Dataset ===
from torchvision.datasets import CIFAR100

DATA_DIR = REPO_DIR / "artifacts/data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Downloading CIFAR-100 to ensure local availability...")
CIFAR100(root=str(DATA_DIR), train=True, download=True)
CIFAR100(root=str(DATA_DIR), train=False, download=True)
print("SUCCESS: CIFAR-100 downloaded and ready.")


In [ ]:
# === CELL 5: Preflight Verification Suite ===
print("Running Table 4 verification suite...")
!python -m pytest tests/test_table4_ablation.py -v

print("Running A0 reuse verification...")
!python scripts/verify_a0_reuse.py


In [ ]:
# === CELL 6: Pilot 2-Task Smoke Test ===
print("Running 2-task pilot test across A1, A2, A3 to guarantee no runtime faults...")
pilot_configs = [
    ("A1_no_ff", "configs/table4_a1_without_ff_cifar100.yaml"),
    ("A2_no_sp", "configs/table4_a2_without_sp_cifar100.yaml"),
    ("A3_no_both", "configs/table4_a3_without_both_cifar100.yaml"),
]
for tag, cfg_path in pilot_configs:
    print(f"\n--- Running Pilot for {tag} ---")
    subprocess.run([
        "python", "scripts/train.py",
        "--config", cfg_path,
        "--max-tasks", "2",
        "--limit-train-per-task", "20",
        "--limit-test-per-task", "10",
        "--device", "cuda",
        "--experiment-type", "pilot",
    ], check=True)
print("\nSUCCESS: Pilot smoke tests completed successfully!")


In [ ]:
# === CELL 7: Inject Reused Condition A0 (Full F-OAL from C3-D) ===
import json

A0_OUTDIR = REPO_DIR / "results/table4_a0_full_foal"
A0_OUTDIR.mkdir(parents=True, exist_ok=True)

a0_summary = {
    "condition": "A0",
    "name": "Full F-OAL (FF On, SP On)",
    "source_run_id": "3v53tmq5",
    "reused_from": "C3-D (Bilinear, Raw [0, 1])",
    "accuracy_bar": 90.4016,
    "a_10": 85.73,
    "f_10_past": 5.16,
    "f_10_code": 4.64,
    "feature_fusion": True,
    "smooth_projection": True,
}
(A0_OUTDIR / "summary.json").write_text(json.dumps(a0_summary, indent=2))
print("A0 (Full F-OAL) successfully injected from C3-D checkpoint:")
print(json.dumps(a0_summary, indent=2))


In [ ]:
# === CELL 8: Full GPU Run A1 — w/o Feature Fusion ===
print("Starting Condition A1: Feature Fusion OFF, Smooth Projection ON (10 Tasks)...")
!python scripts/train.py \
    --config configs/table4_a1_without_ff_cifar100.yaml \
    --feature-cache-dir /content/feature_cache \
    --output-dir results/table4_a1_without_ff \
    --run-name table4_a1_without_ff_seed42 \
    --device cuda \
    --wandb


In [ ]:
# === CELL 9: Full GPU Run A2 — w/o Smooth Projection ===
print("Starting Condition A2: Feature Fusion ON, Smooth Projection OFF (10 Tasks)...")
!python scripts/train.py \
    --config configs/table4_a2_without_sp_cifar100.yaml \
    --feature-cache-dir /content/feature_cache \
    --output-dir results/table4_a2_without_sp \
    --run-name table4_a2_without_sp_seed42 \
    --device cuda \
    --wandb


In [ ]:
# === CELL 10: Full GPU Run A3 — w/o Both (FF Off, SP Off) ===
print("Starting Condition A3: Feature Fusion OFF, Smooth Projection OFF (10 Tasks)...")
!python scripts/train.py \
    --config configs/table4_a3_without_both_cifar100.yaml \
    --feature-cache-dir /content/feature_cache \
    --output-dir results/table4_a3_without_both \
    --run-name table4_a3_without_both_seed42 \
    --device cuda \
    --wandb


In [ ]:
# === CELL 11: Collate Table 4 Ablation Results ===
import json
from pathlib import Path

results = {}
conditions = [
    ("A0 (Full F-OAL)", "results/table4_a0_full_foal/summary.json", 91.1),
    ("A1 (w/o FF)", "results/table4_a1_without_ff/summary.json", 90.6),
    ("A2 (w/o SP)", "results/table4_a2_without_sp/summary.json", 90.7),
    ("A3 (w/o Both)", "results/table4_a3_without_both/summary.json", 90.6),
]

print(f"{'Condition':<20} | {'FF':<5} | {'SP':<5} | {'Paper A_bar':<12} | {'Our A_bar':<12} | {'Delta':<8} | {'A_10':<8} | {'F_10':<8}")
print("-" * 90)

for cond_name, pth, paper_val in conditions:
    p = Path(pth)
    if p.exists():
        data = json.loads(p.read_text())
        a_bar = data.get("accuracy_bar", float("nan"))
        a_10 = data.get("a_10", float("nan"))
        f_10 = data.get("f_10_past", data.get("f_10", float("nan")))
        ff = "ON" if "w/o FF" not in cond_name and "Both" not in cond_name else "OFF"
        sp = "ON" if "w/o SP" not in cond_name and "Both" not in cond_name else "OFF"
        delta = a_bar - paper_val
        print(f"{cond_name:<20} | {ff:<5} | {sp:<5} | {paper_val:6.2f}%     | {a_bar:6.2f}%     | {delta:+6.2f}pp | {a_10:6.2f}%  | {f_10:6.2f}%")
    else:
        print(f"{cond_name:<20} | [NOT FOUND: {pth}]")


In [ ]:
# === CELL 12: Package & Export Results Archive ===
archive_name = "foal_table4_ablation_results.zip"
!zip -r {archive_name} results/table4_* -x "*.pt" "*.pth"
print(f"Archive created: {archive_name}")
try:
    from google.colab import files
    files.download(archive_name)
    print("Triggered automatic browser download.")
except ImportError:
    print("Running outside Google Colab; archive saved locally.")
